# Gurmar preprocessing — product identity

Goal: a **stable product panel** — one `series_id` per real product — so that daily price series are not
broken by renames or merged across different products.

`product_id` only exists from **2026-05-07** onward; earlier files have names only.

Pipeline (aligned with the CPI-weighted web index paper, Sec. III-B and IV-C, and the
`urazkagangunes/InflationResearchStudy` repo):

| Step | What | Paper / repo link |
|---|---|---|
| 0 | Load, parse prices, drop per-kg scraper artifacts (`_1kg`) | III-B: remove scraping artifacts |
| 1 | Exact (100%) name → ID | III-A: match by product IDs / names |
| 2 | One ID, many names — n-gram diffs; validate safe rename types | — |
| 3 | Rule-based 100% matches: truncation, normalization, word order | IV-C: lowercase, strip specials, unit normalization |
| 4 | Pre-ID truncation linking (`...` ↔ full name) | — |
| 5 | Handover assignment: one-to-one, name + pack size + price continuity + time gap | fills IV-C gap: brand/pack-aware matching |
| 6 | Unified name → series map | III-A: stable identifiers |
| 7 | Quality diagnostics: duplicates, >80% jumps, threshold sensitivity, COICOP | IV-A: 80% artifact rule, averaging duplicates |
| 8 | Manual validation sample | VI: "formal manual validation statistics" listed as missing |

Matching is **precision-first**: a wrong link creates fake price jumps; a missed link only shortens a series.

In [1]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":  # paths below are relative to the repository root
    os.chdir(Path.cwd().parent)

import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from scipy.optimize import linear_sum_assignment
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 220)

RAW_DIR = Path("data/raw/_upstream/InflationItems/Datas/Markets/Gurmar")
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ID_START = pd.Timestamp("2026-05-07")
SEED = 42

## 0. Load raw daily files

In [2]:
frames = []
schema_rows = []

for file in sorted(RAW_DIR.glob("gurmar_prices_*.csv")):
    df = pd.read_csv(file, encoding="utf-8-sig", sep=None, engine="python", dtype=str)
    schema_rows.append({"date": file.stem[-10:], "columns": ", ".join(df.columns)})

    df = df.rename(columns={
        "product-id": "product_id",
        "kategori": "category",
        "eski_fiyat": "old_price",
        "tarih": "source_date",
    })
    df["date"] = pd.Timestamp(file.stem[-10:])
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)

raw["name"] = raw["product_name"].astype("string").str.strip()
raw["product_id"] = (
    raw["product_id"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
)
raw["price_raw"] = raw["price"].astype("string").str.strip()

(
    pd.DataFrame(schema_rows)
    .groupby("columns", sort=False)["date"]
    .agg(files="count", first="min", last="max")
    .reset_index()
)

,columns,files,first,last
0,"product_name, price",2,2026-02-21,2026-02-22
1,"product_name, tarih, kategori, price, eski_fiyat, url",1,2026-02-24,2026-02-24
2,"product_name, kategori, price",69,2026-02-26,2026-05-06
3,"product_name, kategori, product_id, price",14,2026-05-07,2026-05-20
4,"product_name, price, product-id",137,2026-05-21,2026-10-04


In [3]:
def parse_price(text):
    if pd.isna(text):
        return np.nan
    text = str(text).replace("₺", "").replace("TL", "").replace("(", "").replace(")", "")
    text = text.split("/")[0].strip().replace(" ", "")
    if "," in text:
        text = text.replace(".", "").replace(",", ".")
    try:
        return float(text)
    except ValueError:
        return np.nan

raw["price"] = raw["price_raw"].map(parse_price)
raw["is_unit_price"] = (
    raw["price_raw"].str.contains("/", na=False)
    | raw["name"].str.contains(r"_1kg$", na=False)
)
raw["is_truncated"] = raw["name"].str.endswith("...", na=False)

unit_rows = raw[raw["is_unit_price"]]
print(
    f"Per-kg artifact rows: {len(unit_rows)} on {unit_rows['date'].nunique()} days "
    f"({unit_rows['date'].min().date()} → {unit_rows['date'].max().date()}); "
    f"base name on the same day: "
    f"{unit_rows.assign(base=unit_rows['name'].str.replace(r'_1kg$', '', regex=True)).merge(raw[['name', 'date']], left_on=['base', 'date'], right_on=['name', 'date']).shape[0]}"
)

Per-kg artifact rows: 351 on 2 days (2026-02-21 → 2026-02-22); base name on the same day: 0


**Decision: drop the `_1kg` rows.** On the first two scrape days the scraper captured the per-kg label
(`(124,90 / Kg)`) instead of the shelf price. Same product, different price basis — keeping them would put
per-kg and per-pack prices in one series (≈2× fake jumps for 500 g items).

In [4]:
raw = raw[~raw["is_unit_price"]].reset_index(drop=True)

n_before = len(raw)
raw = raw.drop_duplicates(["name", "date", "price", "product_id"]).reset_index(drop=True)
print(f"Exact duplicate rows (same name, date, price, ID) dropped: {n_before - len(raw):,} of {n_before:,}")

pre = raw[raw["product_id"].isna()]
post = raw[raw["product_id"].notna()]

pd.Series({
    "rows (after dropping per-kg rows)": len(raw),
    "dates": raw["date"].nunique(),
    "date range": f"{raw['date'].min().date()} → {raw['date'].max().date()}",
    "pre-ID rows (no product_id)": len(pre),
    "pre-ID dates": pre["date"].nunique(),
    "ID-era rows": len(post),
    "ID-era dates": post["date"].nunique(),
    "pre-ID rows that have an ID anyway": int(raw.loc[raw["date"] < ID_START, "product_id"].notna().sum()),
    "ID-era rows missing ID": int(raw.loc[raw["date"] >= ID_START, "product_id"].isna().sum()),
    "unparsed prices": int(raw["price"].isna().sum()),
    "truncated-name rows ('...')": int(raw["is_truncated"].sum()),
}, name="value").to_frame()

Exact duplicate rows (same name, date, price, ID) dropped: 532,614 of 1,089,051


,value
rows (after dropping per-kg rows),556437
dates,221
date range,2026-02-21 → 2026-10-04
pre-ID rows (no product_id),338524
pre-ID dates,70
ID-era rows,217913
ID-era dates,151
pre-ID rows that have an ID anyway,0
ID-era rows missing ID,0
unparsed prices,0


### Scrape coverage per day

Raw files from **2026-05-06 to 2026-08-17** have ~5,100 rows per day but only ~275 distinct products, each
repeated ~19 times (scraper pagination failure upstream). After dropping exact duplicates the gap is visible
below. During this **outage window** only 275 products are observed; for every other product the series has a
hole from 2026-05-04 to 2026-08-18. Time gaps in step 5 are measured *excluding* the outage window.

In [5]:
per_day = raw.groupby("date")["name"].nunique()
pre_median = per_day[per_day.index < ID_START].median()
outage_days = per_day[per_day < 0.5 * pre_median].index
OUTAGE_START, OUTAGE_END = outage_days.min(), outage_days.max()

monthly = per_day.groupby(per_day.index.to_period("M")).agg(["size", "min", "median", "max"])
monthly.columns = ["scrape days", "min products/day", "median products/day", "max products/day"]
print(f"Outage window: {OUTAGE_START.date()} → {OUTAGE_END.date()} ({len(outage_days)} scrape days, "
      f"{(OUTAGE_END - OUTAGE_START).days + 1} calendar days); products/day inside: {int(per_day[outage_days].median())}")
monthly

Outage window: 2026-05-06 → 2026-08-17 (104 scrape days, 104 calendar days); products/day inside: 275


,scrape days,min products/day,median products/day,max products/day
date,,,,
2026-02,6,4648,4928.0,5125
2026-03,30,3954,5007.5,5125
2026-04,29,3908,5008.0,5160
2026-05,30,275,275.0,5138
2026-06,30,274,275.0,275
2026-07,31,275,275.0,275
2026-08,31,275,275.0,4184
2026-09,30,3563,3887.5,4005
2026-10,4,3897,3897.5,3904


In [6]:
outage_names = raw[raw["date"].isin(outage_days)].groupby("name")["date"].nunique()
print(f"Distinct products seen during the outage: {len(outage_names)}; "
      f"seen on ≥90% of outage days: {int((outage_names >= 0.9 * len(outage_days)).sum())}")

def effective_gap(last, nxt):
    """Calendar days between two observations, minus outage days in between."""
    lo, hi = max(last + pd.Timedelta(days=1), OUTAGE_START), min(nxt - pd.Timedelta(days=1), OUTAGE_END)
    outage = max((hi - lo).days + 1, 0)
    return (nxt - last).days - outage

Distinct products seen during the outage: 470; seen on ≥90% of outage days: 157


**Side note — URLs.** The 2026-02-24 file has product URLs with a number (`ananas-adet-4468-p`).
It is a *different* numbering from `product-id`, so it is not used.

In [7]:
url_rows = raw[raw["url"].notna()].copy()
url_rows["url_id"] = url_rows["url"].str.extract(r"-(\d+)-p/?$")[0]

ids_per_name = post.groupby("name")["product_id"].agg(set)
comparable = url_rows[url_rows["name"].isin(ids_per_name.index)]
agree = comparable.apply(lambda r: r["url_id"] in ids_per_name[r["name"]], axis=1)

print(f"URL rows: {len(url_rows)}, comparable to ID era: {len(comparable)}, URL id == product_id: {int(agree.sum())}")

URL rows: 4680, comparable to ID era: 3216, URL id == product_id: 1


## Text and pack-size normalization

Following paper Sec. IV-C: lowercase, remove special characters, standardize whitespace, and convert units
to a common format (`1 LT`, `1L`, `1000 ML` → same). Turkish characters are folded to ASCII because the
store mixes them (`Whıskas` / `Whiskas`).

Pack size is parsed into `(count, size, unit)` with size in **g** or **ml**:
`4*75 Gr` → (4, 75, g); `5 Lt` and `5000 Ml` → (1, 5000, ml); `10'lu` and `10 Adet` → (10, –, –).
Two names are the *same pack* only if these agree. A truncated name (`...`) usually lost its size, so
missing parts act as wildcards for truncated names only.

In [8]:
TR_MAP = str.maketrans("çğıöşüÇĞİÖŞÜ", "cgiosuCGIOSU")

def fold(text):
    """Lowercase + Turkish characters to ASCII + collapse whitespace."""
    text = unicodedata.normalize("NFKC", str(text)).replace("İ", "i").replace("I", "ı")
    text = text.lower().translate(TR_MAP)
    return re.sub(r"\s+", " ", text).strip()

UNIT_PATTERNS = [
    (r"(\d),(\d)", r"\1.\2"),                    # 1,5 -> 1.5
    (r"(\d)\s*[x\*]\s*(\d)", r"\1x\2"),          # 4*115 / 4 x 115 -> 4x115
    (r"\b(gr|grm|gram)\b\.?", "g"),
    (r"\b(lt|litre)\b\.?", "l"),
    (r"\bml\b\.?", "ml"),
    (r"\b(adet|ad)\b\.?", "adet"),
    (r"(\d)\s+(g|kg|ml|l|cl|adet)\b", r"\1\2"),  # 500 g -> 500g
    (r"'\s*(li|lu)\b", "li"),                    # 10'lu -> 10li
    (r"(\d)\s*(li|lu)\b", r"\1li"),
]

def normalize(text):
    """fold() + unit/number formatting + punctuation removal."""
    text = fold(re.sub(r"\.\.\.$", "", str(text)))
    for pattern, repl in UNIT_PATTERNS:
        text = re.sub(pattern, repl, text)
    text = re.sub(r"[^\w\.\s]", " ", text)
    text = re.sub(r"(?<!\d)\.|\.(?!\d)", " ", text)
    return re.sub(r"\s+", " ", text).strip()

UNIT_TO_BASE = {"kg": ("g", 1000), "g": ("g", 1), "gr": ("g", 1), "mg": ("g", 0.001),
                "l": ("ml", 1000), "lt": ("ml", 1000), "ml": ("ml", 1), "cl": ("ml", 10)}
SIZE = r"(\d+(?:\.\d+)?)\s*(kg|gr|g|mg|lt|l|ml|cl)\b"

def parse_pack(name):
    """(count, size, unit) with size in g or ml; None where not stated."""
    text = fold(re.sub(r"\.\.\.$", "", str(name)))
    text = re.sub(r"(\d),(\d)", r"\1.\2", text)
    count = size = unit = None

    multi = re.search(r"(\d+)\s*[x\*]\s*" + SIZE, text)
    if multi:
        count = int(multi.group(1))
        size, unit = float(multi.group(2)), multi.group(3)
    else:
        sizes = re.findall(SIZE, text)
        if sizes:
            size, unit = float(sizes[-1][0]), sizes[-1][1]
        pieces = re.search(r"(\d+)\s*(?:'\s*)?(li|lu|lik|luk|adet|ad)\b", text)
        if pieces:
            count = int(pieces.group(1))

    if unit is not None:
        base, factor = UNIT_TO_BASE[unit]
        size, unit = round(size * factor, 3), base
    return (count, size, unit)

def same_pack(name_a, name_b):
    """Pack sizes agree; for a truncated name, its missing parts are wildcards."""
    (c_a, s_a, u_a), (c_b, s_b, u_b) = parse_pack(name_a), parse_pack(name_b)
    trunc_a, trunc_b = str(name_a).endswith("..."), str(name_b).endswith("...")

    def agree(x, y):
        if x == y:
            return True
        if x is None and trunc_a:
            return True
        if y is None and trunc_b:
            return True
        return False

    count_ok = agree(c_a or (1 if s_a else None), c_b or (1 if s_b else None)) or agree(c_a, c_b)
    return agree(s_a, s_b) and agree(u_a, u_b) and count_ok

examples = ["Coca Cola 4*1 Lt", "Bingo Soft 5000 Ml", "Soft Kuzumun Kokusu 5 Lt", "Bulaşık Süngeri 5'li",
            "Bulaşık Süngeri Oluklu 5 Adet", "Uysal Ezine Peyn - 500 Gr.", "Omo Toz Deterjan 4,5 ...",
            "Ton Balık 3*75 Gr", "Dardanel Ton Balık 4*75 Gr"]
pd.DataFrame({"name": examples, "pack": [parse_pack(e) for e in examples]})

,name,pack
0,Coca Cola 4*1 Lt,"(4, 1000.0, ml)"
1,Bingo Soft 5000 Ml,"(None, 5000.0, ml)"
2,Soft Kuzumun Kokusu 5 Lt,"(None, 5000.0, ml)"
3,Bulaşık Süngeri 5'li,"(5, None, None)"
4,Bulaşık Süngeri Oluklu 5 Adet,"(5, None, None)"
5,Uysal Ezine Peyn - 500 Gr.,"(None, 500.0, g)"
6,"Omo Toz Deterjan 4,5 ...","(None, None, None)"
7,Ton Balık 3*75 Gr,"(3, 75.0, g)"
8,Dardanel Ton Balık 4*75 Gr,"(4, 75.0, g)"


In [9]:
pd.DataFrame([
    ("Bingo Soft 5000 Ml", "Soft Kuzumun Kokusu 5 Lt"),
    ("Bulaşık Süngeri 5'li", "Bulaşık Süngeri Oluklu 5 Adet"),
    ("Namet Hindi Uzun Sosis 420 Gr", "Namet Hindi Uzun Sosis 210 Gr"),
    ("Ton Balık 3*75 Gr", "Dardanel Ton Balık 4*75 Gr"),
    ("Omo Active Oxygen Toz Deterjan 4,5 ...", "Omo Active Oxygen Toz Deterjan 4,5 Kg"),
], columns=["a", "b"]).assign(same_pack=lambda d: [same_pack(a, b) for a, b in zip(d["a"], d["b"])])

,a,b,same_pack
0,Bingo Soft 5000 Ml,Soft Kuzumun Kokusu 5 Lt,True
1,Bulaşık Süngeri 5'li,Bulaşık Süngeri Oluklu 5 Adet,True
2,Namet Hindi Uzun Sosis 420 Gr,Namet Hindi Uzun Sosis 210 Gr,False
3,Ton Balık 3*75 Gr,Dardanel Ton Balık 4*75 Gr,False
4,"Omo Active Oxygen Toz Deterjan 4,5 ...","Omo Active Oxygen Toz Deterjan 4,5 Kg",True


## 1. Exact (100%) string match: pre-ID name → ID

A pre-ID name is mapped when the **identical** string appears in the ID era with exactly one ID.
Names whose string carries two IDs in the ID era are **not** mapped here (paper IV-C: exclude what cannot be
matched reliably); they go to the handover assignment in step 5, where price continuity decides.

In [10]:
post_name_ids = post.groupby("name")["product_id"].agg(lambda s: sorted(set(s)))
post_name_n_ids = post_name_ids.map(len)

pre_names = (
    pre.groupby("name")
    .agg(rows=("date", "size"), first_seen=("date", "min"), last_seen=("date", "max"),
         is_truncated=("is_truncated", "max"))
    .reset_index()
)

pre_names["n_ids"] = pre_names["name"].map(post_name_n_ids).fillna(0).astype(int)
pre_names["step1"] = np.select(
    [pre_names["n_ids"] == 1, pre_names["n_ids"] > 1],
    ["exact_unique", "exact_ambiguous"],
    default="no_exact_match",
)

step1_map = (
    pre_names[pre_names["step1"] == "exact_unique"][["name"]]
    .assign(series_id=lambda d: d["name"].map(post_name_ids).map(lambda ids: ids[0]), method="1_exact")
)

(
    pre_names.groupby("step1")
    .agg(names=("name", "size"), rows=("rows", "sum"))
    .assign(pct_names=lambda d: 100 * d["names"] / d["names"].sum(),
            pct_rows=lambda d: 100 * d["rows"] / d["rows"].sum())
    .round(1)
)

,names,rows,pct_names,pct_rows
step1,,,,
exact_ambiguous,7,548,0.1,0.2
exact_unique,4749,272720,69.5,80.6
no_exact_match,2076,65256,30.4,19.3


In [11]:
print(f"ID-era names carrying >1 ID: {(post_name_n_ids > 1).sum()} of {len(post_name_n_ids)}")
post_name_ids[post_name_n_ids > 1].head(10)

ID-era names carrying >1 ID: 19 of 6678


name
Ariel Toz Çamaşır Deterjanı Dağ Esintisi Renklilere Özel 1.5 Kg      ['4504' '9366']
Ayçiçek Yağı 1 Lt                                                    ['1554' '5355']
Bebe Sütlü Bisküvi 172 Gr                                           ['10607' '8037']
Dardanel Kolay Pişir Tereyağlı Jumbo Karides 250 Gr                ['11483' '13536']
Dardanel Ton Balığı 4*75 Gr                                          ['3411' '5224']
Duru Natural Olive Zeytinyağı Özlü Banyo Sabunu 4'lü 150 Gr           ['1157' '944']
Ekmek 200 Gr                                                         ['5646' '8620']
Huggies Drynites Gece Külodu (4-7 yaş) 10'lu                          ['1735' '223']
Köftelik Bulgur 1 Kg                                                  ['309' '8529']
Nescafe 3'ü 1 Arada Gold Yoğun İçim                                ['12910' '12927']
Name: product_id, dtype: list<item: string>[pyarrow]

## 2. One ID, many names

Inside the ID era some IDs show several names over time. Names per ID are ordered by first appearance and each
consecutive pair (old → new) is compared: word diff, word-bigram / char-3-gram Jaccard, rule-based diff type,
and whether the **pack size** changed under the same ID.

In [12]:
def ngrams(tokens, n):
    return {tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)}

def jaccard(a, b):
    return len(a & b) / len(a | b) if (a or b) else 1.0

def char_ngrams(text, n=3):
    text = f" {text} "
    return {text[i:i + n] for i in range(len(text) - n + 1)}

def classify_pair(old, new):
    if old == new:
        return "identical"
    if fold(old) == fold(new):
        return "case/whitespace/Turkish chars"
    n_old, n_new = normalize(old), normalize(new)
    if n_old == n_new:
        return "unit/number formatting"
    t_old, t_new = set(n_old.split()), set(n_new.split())
    if t_old == t_new:
        return "word order"
    if t_old < t_new:
        return "words added (e.g. brand/descriptor)"
    if t_new < t_old:
        return "words removed"
    return "rewrite (words added and removed)"

def pack_changed(old, new):
    (c_o, s_o, u_o), (c_n, s_n, u_n) = parse_pack(old), parse_pack(new)
    size_diff = s_o is not None and s_n is not None and (s_o, u_o) != (s_n, u_n)
    count_diff = c_o is not None and c_n is not None and c_o != c_n
    return size_diff or count_diff

name_spans = (
    post.groupby(["product_id", "name"])["date"]
    .agg(first_seen="min", last_seen="max", days="nunique")
    .reset_index()
    .sort_values(["product_id", "first_seen"])
)
names_per_id = name_spans.groupby("product_id")["name"].size()
multi_ids = names_per_id[names_per_id > 1].index
same_day = post.groupby(["product_id", "date"])["name"].nunique()

pair_rows = []
for pid, group in name_spans[name_spans["product_id"].isin(multi_ids)].groupby("product_id"):
    names, firsts = group["name"].tolist(), group["first_seen"].tolist()
    for old, new, switch in zip(names, names[1:], firsts[1:]):
        t_old, t_new = normalize(old).split(), normalize(new).split()
        pair_rows.append({
            "product_id": pid, "old_name": old, "new_name": new, "switch_date": switch,
            "removed": " ".join(sorted(set(t_old) - set(t_new))),
            "added": " ".join(sorted(set(t_new) - set(t_old))),
            "word1_jaccard": jaccard(set(t_old), set(t_new)),
            "word2_jaccard": jaccard(ngrams(t_old, 2), ngrams(t_new, 2)),
            "char3_jaccard": jaccard(char_ngrams(normalize(old)), char_ngrams(normalize(new))),
            "diff_type": classify_pair(old, new),
            "pack_size_changed": pack_changed(old, new),
        })
pairs = pd.DataFrame(pair_rows)

pd.Series({
    "IDs in ID era": len(names_per_id),
    "IDs with 1 name": int((names_per_id == 1).sum()),
    "IDs with 2 names": int((names_per_id == 2).sum()),
    "IDs with 3 names": int((names_per_id == 3).sum()),
    "IDs with 4+ names": int((names_per_id >= 4).sum()),
    "ID-days with >1 name on the same day": int((same_day > 1).sum()),
    "rename pairs (old → new)": len(pairs),
    "pairs where pack size changed under the same ID": int(pairs["pack_size_changed"].sum()),
}, name="value").to_frame()

,value
IDs in ID era,5757
IDs with 1 name,5133
IDs with 2 names,321
IDs with 3 names,290
IDs with 4+ names,13
ID-days with >1 name on the same day,0
rename pairs (old → new),940
pairs where pack size changed under the same ID,111


In [13]:
(
    pairs.groupby("diff_type")
    .agg(pairs=("product_id", "size"),
         pack_changed=("pack_size_changed", "sum"),
         mean_char3=("char3_jaccard", "mean"),
         mean_word1=("word1_jaccard", "mean"))
    .assign(pct=lambda d: 100 * d["pairs"] / d["pairs"].sum())
    .sort_values("pairs", ascending=False)
    .round(3)
)

,pairs,pack_changed,mean_char3,mean_word1,pct
diff_type,,,,,
rewrite (words added and removed),533,111,0.485,0.435,56.702
words added (e.g. brand/descriptor),214,0,0.759,0.759,22.766
words removed,141,0,0.659,0.696,15.000
unit/number formatting,29,0,1.000,1.000,3.085
case/whitespace/Turkish chars,16,0,1.000,1.000,1.702
word order,7,0,0.881,1.000,0.745


In [14]:
bins = [0, 0.3, 0.5, 0.7, 0.9, 1.0001]
labels = ["<0.3", "0.3-0.5", "0.5-0.7", "0.7-0.9", "0.9-1.0"]

pd.DataFrame({
    col: pd.cut(pairs[col], bins, labels=labels, right=False).value_counts().sort_index()
    for col in ["char3_jaccard", "word1_jaccard", "word2_jaccard"]
})

,char3_jaccard,word1_jaccard,word2_jaccard
<0.3,122,147,420
0.3-0.5,175,152,120
0.5-0.7,257,279,201
0.7-0.9,321,309,154
0.9-1.0,65,53,45


In [15]:
removed_tokens = pairs["removed"].str.split().explode().dropna().value_counts().head(15)
added_tokens = pairs["added"].str.split().explode().dropna().value_counts().head(15)

pd.DataFrame({
    "removed_token": removed_tokens.index, "removed_n": removed_tokens.values,
    "added_token": added_tokens.index, "added_n": added_tokens.values,
})

,removed_token,removed_n,added_token,added_n
0,ulker,25,ulker,50
1,eti,23,eti,19
2,kokulu,20,paket,19
3,nestle,18,nestle,15
4,knorr,17,knorr,15
5,suyu,16,solen,13
6,12li,15,koli,11
7,cikolata,14,12li,11
8,paket,13,dr,10
9,koli,13,oetker,10


In [16]:
renames_by_day = pairs["switch_date"].dt.date.value_counts().sort_index()
renames_by_day[renames_by_day >= 10].to_frame("renames on date (days with 10+)")

,renames on date (days with 10+)
switch_date,
2026-08-28,347
2026-09-01,25
2026-09-05,18
2026-09-16,33
2026-09-17,76
2026-09-18,10
2026-09-19,59
2026-09-22,51
2026-09-23,114


### 2a. Safe rename types — full inspection

Candidate rules for step 3: *case/whitespace/Turkish chars*, *unit/number formatting*, *word order*.
All pairs of these types are listed (not a sample) so the rules can be approved before use.

In [17]:
SAFE_TYPES = ["case/whitespace/Turkish chars", "unit/number formatting", "word order"]
safe_pairs = pairs[pairs["diff_type"].isin(SAFE_TYPES)].sort_values(["diff_type", "product_id"])

print(safe_pairs["diff_type"].value_counts().to_string())
print(f"pack size changed inside safe types: {int(safe_pairs['pack_size_changed'].sum())}")
safe_pairs[["diff_type", "product_id", "old_name", "new_name"]]

diff_type
unit/number formatting           29
case/whitespace/Turkish chars    16
word order                        7
pack size changed inside safe types: 0


,diff_type,product_id,old_name,new_name
7,case/whitespace/Turkish chars,10026,OVİL EV TİPİ KIRMA ZEYTİN - 500 Gr.,Ovil Ev Tipi Kırma Zeytin - 500 Gr.
72,case/whitespace/Turkish chars,10964,Whıskas Kuzulu Yaş Mama 85 GR,Whiskas Kuzulu Yaş Mama 85 Gr
73,case/whitespace/Turkish chars,10965,Whıskas Kuzulu Hindili Mama 85 Gr,Whiskas Kuzulu Hindili Mama 85 Gr
74,case/whitespace/Turkish chars,10966,Whıskas Somonlu-Havuçlu Mama 85 Gr,Whiskas Somonlu-Havuçlu Mama 85 Gr
259,case/whitespace/Turkish chars,1705,Nestle Harfler Çikolatalı Tahıl Gevreği 310 gr,Nestle Harfler Çikolatalı Tahıl Gevreği 310 Gr
269,case/whitespace/Turkish chars,1781,Filiz Burgu Makarna 500 gr,Filiz Burgu Makarna 500 Gr
322,case/whitespace/Turkish chars,2191,Filiz Çorbalik Tel Şehriye 500 gr,Filiz Çorbalik Tel Şehriye 500 Gr
430,case/whitespace/Turkish chars,4038,Taft Power Kaşmir Mega Güclü Saç Köpüğü 150 Ml,Taft Power Kaşmir Mega Güçlü Saç Köpüğü 150 ml
431,case/whitespace/Turkish chars,4041,Taft Power Kaşmir Mega Güclü Saç Spreyi 250 Ml,Taft Power Kaşmir Mega Güçlü Saç Spreyi 250 ml
504,case/whitespace/Turkish chars,5665,Neutrogena Yoğun Bakım Kremi 200 Ml,Neutrogena Yoğun Bakım Kremi 200 ml


In [18]:
pd.set_option("display.max_rows", 200)
pairs.loc[pairs["pack_size_changed"], ["product_id", "diff_type", "old_name", "new_name", "switch_date"]].head(20)

,product_id,diff_type,old_name,new_name,switch_date
16,10130,rewrite (words added and removed),Tuzlu Çubuk Kraker 32 Gr,Ülker Tuzlu Çubuk Kraker 40 Gr,2026-10-02
19,10147,rewrite (words added and removed),Parex Klasik Yeşil Bulaşık Süngeri 5'li,Klasik Yeşil Düz Bulaşık Süngeri 10 Adet,2026-08-28
20,10147,rewrite (words added and removed),Klasik Yeşil Düz Bulaşık Süngeri 10 Adet,Parex Klasik Yeşil Düz Bulaşık Süngeri 5Li,2026-09-04
55,10607,rewrite (words added and removed),Ülker Bebe Bisküvisi 400 Gr,Bebe Sütlü Bisküvi 172 Gr,2026-08-28
56,10607,rewrite (words added and removed),Bebe Sütlü Bisküvi 172 Gr,Ülker Bebe Sütlü Bisküvi 400 Gr,2026-09-23
59,10631,rewrite (words added and removed),9 Kat Ince Ince Fındıklı 144 gr,Ülker 9 Kat Ince Ince Fındıklı 114 Gr,2026-10-02
61,10722,rewrite (words added and removed),Halley Çikolata Bisküvi 8'li,Halley Çikolata Bisküvi 7'li,2026-08-21
65,10837,rewrite (words added and removed),Ace Ultra Kıvamlı Dağ Ferahlığı Çamaşır Suyu 810 Gr,Ultra Power Ferahlatıcı Etkili Jel 750ml - 12'li Koli,2026-08-28
66,10837,rewrite (words added and removed),Ultra Power Ferahlatıcı Etkili Jel 750ml - 12'li Koli,Ace Ultra Power Ferahlatıcı Çamaşır Suyu Etkili Jel 810 Gr,2026-09-18
68,10892,rewrite (words added and removed),Şeftali Muz Kavanoz Mama 125 gr,Hero Şeftali Muz Kavanoz Mama 120 Gr,2026-09-17


## 3. Rule-based 100% matches for names with no exact match

Candidates are **every name any ID ever had** (the store renames products). Rules, applied in order:

- `3b_truncated_prefix`: name ends with `...`; its text is a prefix of exactly one ID's name(s), pack compatible
- `3c_normalized`: identical after `normalize()` (case, Turkish chars, units, punctuation)
- `3d_word_order`: same word set after `normalize()`, different order (validated in 2a)

A rule that finds more than one ID is marked `_ambiguous` and the name moves on to step 5.
The former `_1kg` suffix rule is gone (rows dropped in step 0).

In [19]:
candidates = name_spans[["product_id", "name"]].drop_duplicates().copy()
candidates["norm"] = candidates["name"].map(normalize)
candidates["word_key"] = candidates["norm"].str.split().map(lambda t: " ".join(sorted(t)))

norm_lookup = candidates.groupby("norm")["product_id"].agg(lambda s: sorted(set(s)))
word_lookup = candidates.groupby("word_key")["product_id"].agg(lambda s: sorted(set(s)))

def match_rules(name):
    if name.endswith("..."):
        prefix = name[:-3].rstrip()
        hit = candidates[candidates["name"].str.startswith(prefix)]
        if hit.empty:
            hit = candidates[candidates["norm"].str.startswith(normalize(prefix))]
        hit = hit.loc[np.array([same_pack(name, n) for n in hit["name"]], dtype=bool)]
        ids = hit["product_id"].unique()
        if len(ids):
            return ("3b_truncated_prefix", ids[0] if len(ids) == 1 else None, len(ids))

    norm = normalize(name)
    ids = norm_lookup.get(norm)
    if ids is not None:
        return ("3c_normalized", ids[0] if len(ids) == 1 else None, len(ids))

    ids = word_lookup.get(" ".join(sorted(norm.split())))
    if ids is not None:
        return ("3d_word_order", ids[0] if len(ids) == 1 else None, len(ids))

    return ("no_rule_match", None, 0)

no_exact = pre_names[pre_names["step1"] == "no_exact_match"].copy()
no_exact[["rule", "series_id", "n_candidate_ids"]] = pd.DataFrame(
    no_exact["name"].map(match_rules).tolist(), index=no_exact.index
)
no_exact["rule"] = np.where(
    (no_exact["rule"] != "no_rule_match") & no_exact["series_id"].isna(),
    no_exact["rule"] + "_ambiguous", no_exact["rule"],
)

step3_map = no_exact.loc[no_exact["series_id"].notna(), ["name", "series_id", "rule"]].rename(columns={"rule": "method"})

(
    no_exact.groupby("rule")
    .agg(names=("name", "size"), rows=("rows", "sum"))
    .assign(pct_names=lambda d: 100 * d["names"] / len(no_exact),
            pct_rows=lambda d: 100 * d["rows"] / no_exact["rows"].sum())
    .round(1)
)

,names,rows,pct_names,pct_rows
rule,,,,
3b_truncated_prefix,554,555,26.7,0.9
3b_truncated_prefix_ambiguous,30,57,1.4,0.1
3c_normalized,64,3480,3.1,5.3
3d_word_order,9,467,0.4,0.7
no_rule_match,1419,60697,68.4,93.0


## 4. Pre-ID truncation linking

Leftover names that end with `...` are linked to a leftover **full** name from the pre-ID era when the full
name is the only one starting with that prefix, pack sizes are compatible and the two never appear on the same
day. Both then form one *group* that moves together into step 5.

In [20]:
leftover_names = pd.concat([
    pre_names.loc[pre_names["step1"] == "exact_ambiguous", "name"],
    no_exact.loc[no_exact["series_id"].isna(), "name"],
]).tolist()

days_by_name = pre.groupby("name")["date"].agg(lambda s: set(s))
full_names = [n for n in leftover_names if not n.endswith("...")]
full_norm = pd.Series({n: normalize(n) for n in full_names})

group_of = {n: n for n in leftover_names}
trunc_links = []
for name in (n for n in leftover_names if n.endswith("...")):
    prefix = normalize(name)
    hits = full_norm[full_norm.str.startswith(prefix)].index
    hits = [h for h in hits if same_pack(name, h) and not (days_by_name[name] & days_by_name[h])]
    if len(hits) == 1:
        group_of[name] = hits[0]
        trunc_links.append((name, hits[0]))

groups = pd.Series(group_of).rename_axis("name").reset_index(name="group")
print(f"Leftover names: {len(leftover_names)}  truncated: {sum(n.endswith('...') for n in leftover_names)}  "
      f"linked to a full name: {len(trunc_links)}  groups: {groups['group'].nunique()}")
pd.DataFrame(trunc_links, columns=["truncated", "full_name"]).head(8)

Leftover names: 1456  truncated: 205  linked to a full name: 166  groups: 1290


,truncated,full_name
0,Air Wick Elektirikli Kit Oda Kokusu Temizlik Esin...,Air Wick Elektirikli Kit Oda Kokusu Temizlik Esintisi 19 Ml
1,Air Wick Elektirikli Kit Oda Kokusu Yedek Bahar Ç...,Air Wick Elektirikli Kit Oda Kokusu Yedek Bahar Çiçekleri 19 Ml
2,Air Wick Elektirikli Kit Oda Kokusu Yedek Lavanta...,Air Wick Elektirikli Kit Oda Kokusu Yedek Lavanta 19 Ml
3,Air Wick Eliktirikli Kit Oda Kokusu Yedek Temizli...,Air Wick Eliktirikli Kit Oda Kokusu Yedek Temizlik Esintisi 19 Ml
4,Air Wick Freshmatic Bahar Çiçekleri Kokulu Oda Ko...,Air Wick Freshmatic Bahar Çiçekleri Kokulu Oda Kokusu Yedek Sprey 250 Ml
5,Air Wick Freshmatic Bahar Çiçekleri Yedekli Oda K...,Air Wick Freshmatic Bahar Çiçekleri Yedekli Oda Kokusu
6,Algida Gökkuşağı Meyveli&Patlayan Şekerli Dondrum...,Algida Gökkuşağı Meyveli&Patlayan Şekerli Dondruma 62 ML
7,Algida Magnum Chill Yaban Mersini Cookie Dondurma...,Algida Magnum Chill Yaban Mersini Cookie Dondurma 85 Ml


## 5. Handover assignment (one-to-one)

Each leftover group is an *old* series; each ID is a *new* series. A link is a one-to-one assignment, not a
cluster: an old series can continue as at most one ID and an ID absorbs at most one leftover group.

**Eligible pair** (old group *g*, ID *i*):
1. *Time:* *i* has no observation inside *g*'s active window, and has at least one after *g*'s last day.
2. *Blocking:* *i* is among the 25 nearest IDs by TF-IDF char 3–5-gram cosine on normalized names.
3. *Pack:* same pack size (`same_pack`).
4. *Price continuity:* next price of *i* after the gap / last price of *g* ∈ [0.80, 1.25].
5. *Name:* `token_sort_ratio` ≥ 70 if the gap is ≤ 14 days, ≥ 85 if longer; and `token_guard`: every word that
   differs must be a spelling variant of a word on the other side, or a brand (first word) that was added or
   dropped. This rejects flavour/variant changes (`Çilekli` → `Çikolatalı`, `Islak Havlu` → `Çocuk Bezi`)
   and brand swaps (`Meysu` → `Dimes`).
6. *Gap* is measured excluding the outage window (`effective_gap`).

**Score** = 0.6 · name similarity + 0.4 · price closeness (1 at equal price, 0 at the ±25 % edge).
`scipy.optimize.linear_sum_assignment` picks the score-maximizing one-to-one set.

In [21]:
MAX_PRICE_RATIO = 1.25
SHORT_GAP_DAYS = 14
NAME_MIN_SHORT_GAP = 70
NAME_MIN_LONG_GAP = 85
TOP_K = 25

GENERIC_TOKENS = {"adet", "paket", "yeni", "kg", "g", "ml", "l", "li", "koli", "ekonomik", "avantaj", "firsat",
                  "eko", "boy", "buyuk", "kucuk", "ve", "ile", "icin"}

def variant_tokens(name):
    tokens = normalize(name).split()
    return [t for t in tokens if not re.search(r"\d", t) and t not in GENERIC_TOKENS]

def token_guard(old, new):
    """Differing words must be spelling variants or a brand (first word); flavour/variant changes fail."""
    t_old, t_new = variant_tokens(old), variant_tokens(new)
    if not t_old or not t_new:
        return False
    if t_old[0] != t_new[0] and t_old[0] not in t_new and t_new[0] not in t_old:
        return False  # brand swap (Meysu → Dimes, Ernet → Piknik)
    brands = {t_old[0], t_new[0]}
    only_old = [t for t in set(t_old) - set(t_new) if t not in brands]
    only_new = [t for t in set(t_new) - set(t_old) if t not in brands]

    def has_variant(t, pool):
        return any(fuzz.ratio(t, u) >= 80 or u.startswith(t) or t.startswith(u) for u in pool)

    return all(has_variant(t, only_new) for t in only_old) and all(has_variant(t, only_old) for t in only_new)

# observations already assigned to real IDs (ID era + steps 1 and 3)
linked_pre = pre.merge(pd.concat([step1_map, step3_map])[["name", "series_id"]], on="name")
id_obs = pd.concat([
    post[["product_id", "date", "price"]].rename(columns={"product_id": "series_id"}),
    linked_pre[["series_id", "date", "price"]],
]).sort_values(["series_id", "date"])
id_dates = id_obs.groupby("series_id")["date"].apply(lambda s: np.sort(s.unique()))
id_obs_idx = id_obs.drop_duplicates(["series_id", "date"]).set_index(["series_id", "date"])["price"]

# old groups
g_obs = pre.merge(groups, on="name").sort_values("date")
g_info = g_obs.groupby("group").agg(first=("date", "min"), last=("date", "max"), rows=("date", "size"))
g_info["last_price"] = g_obs.groupby("group")["price"].last()

# blocking by name similarity
id_names = candidates.groupby("product_id")["norm"].apply(list)
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit(candidates["norm"].tolist() + [normalize(g) for g in g_info.index])
cand_matrix = vectorizer.transform(candidates["norm"])
knn = NearestNeighbors(n_neighbors=TOP_K * 3, metric="cosine").fit(cand_matrix)
_, nn_idx = knn.kneighbors(vectorizer.transform([normalize(g) for g in g_info.index]))

edges = []
for row, group in enumerate(g_info.index):
    info = g_info.loc[group]
    seen = set()
    for idx in nn_idx[row]:
        pid, cand_name = candidates["product_id"].iat[idx], candidates["name"].iat[idx]
        if pid in seen:
            continue
        seen.add(pid)
        if len(seen) > TOP_K:
            break
        dates = id_dates.get(pid)
        if dates is None or ((dates >= info["first"]) & (dates <= info["last"])).any() or (dates < info["first"]).any():
            continue
        after = dates[dates > info["last"]]
        if len(after) == 0 or not same_pack(group, cand_name):
            continue
        next_date = after[0]
        ratio = id_obs_idx[(pid, next_date)] / info["last_price"]
        if not (1 / MAX_PRICE_RATIO <= ratio <= MAX_PRICE_RATIO):
            continue
        gap = effective_gap(info["last"], next_date)
        sim = max(fuzz.token_sort_ratio(normalize(group), n) for n in id_names[pid])
        if sim < (NAME_MIN_SHORT_GAP if gap <= SHORT_GAP_DAYS else NAME_MIN_LONG_GAP):
            continue
        if not token_guard(group, cand_name):
            continue
        closeness = 1 - abs(np.log(ratio)) / np.log(MAX_PRICE_RATIO)
        edges.append({"group": group, "series_id": pid, "candidate_name": cand_name, "gap_days": gap,
                      "price_ratio": ratio, "name_sim": sim, "score": 0.6 * sim / 100 + 0.4 * closeness})

edges = pd.DataFrame(edges).sort_values("score", ascending=False).drop_duplicates(["group", "series_id"])
print(f"Old groups: {len(g_info)}  eligible edges: {len(edges)}  groups with ≥1 edge: {edges['group'].nunique()}")

Old groups: 1290  eligible edges: 23  groups with ≥1 edge: 19


In [22]:
g_index = {g: k for k, g in enumerate(edges["group"].unique())}
i_index = {i: k for k, i in enumerate(edges["series_id"].unique())}
cost = np.full((len(g_index), len(i_index)), 1e6)
for e in edges.itertuples():
    cost[g_index[e.group], i_index[e.series_id]] = -e.score

rows, cols = linear_sum_assignment(cost)
inv_g, inv_i = list(g_index), list(i_index)
chosen = {(inv_g[r], inv_i[c]) for r, c in zip(rows, cols) if cost[r, c] < 1e5}

assigned = edges[[(g, i) in chosen for g, i in zip(edges["group"], edges["series_id"])]].copy()
assigned["tier"] = np.where(assigned["gap_days"] <= SHORT_GAP_DAYS, "5_handover_short_gap", "5_handover_long_gap")

pd.DataFrame({
    "groups": assigned["tier"].value_counts(),
    "median_name_sim": assigned.groupby("tier")["name_sim"].median(),
    "median_effective_gap_days": assigned.groupby("tier")["gap_days"].median(),
    "median_abs_price_change_%": assigned.groupby("tier")["price_ratio"].apply(lambda r: (100 * (r - 1).abs()).median()).round(2),
    "exact_same_price_share": assigned.groupby("tier")["price_ratio"].apply(lambda r: np.isclose(r, 1).mean()).round(3),
})

,groups,median_name_sim,median_effective_gap_days,median_abs_price_change_%,exact_same_price_share
tier,,,,,
5_handover_long_gap,7,97.222222,16.0,0.00,0.714
5_handover_short_gap,12,93.774219,2.0,2.78,0.500


In [23]:
show = assigned.sample(frac=1, random_state=SEED).groupby("tier").head(8).sort_values("tier")
show[["tier", "group", "candidate_name", "series_id", "gap_days", "price_ratio", "name_sim"]].round(3)

,tier,group,candidate_name,series_id,gap_days,price_ratio,name_sim
1,5_handover_long_gap,Dardanel Kolay Pişir Tereyağlı Jumbo Karides 250 Gr,Dardanel Kolay Pişir Tereyağlı Jumbo Karides 250 Gr,11483,16,1.000,100.000
4,5_handover_long_gap,Ekmek 200 Gr,Ekmek 200 Gr,8620,33,1.000,100.000
22,5_handover_long_gap,Ülker Rondo Çilekli&Kremalı Bisküvi 61 Gr,Rondo Kremalı Çilekli Bisküvi 61 gr,8416,20,1.000,91.667
17,5_handover_long_gap,Ramazan Kolisi,Ramazan Kolisi,12147,15,1.000,100.000
16,5_handover_long_gap,Oneo Slims 60 Dk Meyve Aromalı Sakız 14 Gr,Ülker Oneo Slims 60 Dk Meyve Aromalı Sakız 14 Gr,12486,47,1.149,93.023
10,5_handover_long_gap,Knorr Kıtır Domates Çabuk Çorba 22 Gr,Knorr Kıtırlı Domates Çabuk Çorba (22 g),7049,16,1.236,97.222
9,5_handover_long_gap,Jacobs 3ü 1 Arada 16 Gr,Jacobs 3ü1 Arada 16 Gr,3298,16,1.000,92.683
20,5_handover_short_gap,Yöresel Mihaliç Peyn - 500 Gr.,Yöresel Mihaliç Peyniri - 500 Gr.,4819,1,1.000,94.340
12,5_handover_short_gap,La Lorraine Pizza Çörek,La Lorraine Pizza Çörek Adet,12145,2,1.083,90.196
11,5_handover_short_gap,La Lorraine Balkan Çöreği,La Lorraine Balkan Çöreği Adet,11818,2,1.137,90.909


In [24]:
unassigned = g_info.drop(index=assigned["group"])
near_switch = unassigned["last"] >= ID_START - pd.Timedelta(days=7)
print(f"Unassigned groups: {len(unassigned)} ({int(unassigned['rows'].sum())} rows); "
      f"last seen in the final pre-ID week: {int(near_switch.sum())}")
unassigned["last"].dt.to_period("M").value_counts().sort_index().to_frame("unassigned groups by last-seen month")

Unassigned groups: 1271 (60175 rows); last seen in the final pre-ID week: 839


,unassigned groups by last-seen month
last,
2026-02,84
2026-03,188
2026-04,160
2026-05,839


## 6. Unified name → series map

In [25]:
group_series = assigned.set_index("group")["series_id"]
group_method = assigned.set_index("group")["tier"]

leftover_map = groups.copy()
leftover_map["series_id"] = leftover_map["group"].map(group_series)
leftover_map["method"] = leftover_map["group"].map(group_method)

own = leftover_map["series_id"].isna()
own_ids = {g: f"n_{k:04d}" for k, g in enumerate(sorted(leftover_map.loc[own, "group"].unique()))}
leftover_map.loc[own, "series_id"] = leftover_map.loc[own, "group"].map(own_ids)
leftover_map.loc[own, "method"] = "6_own_series"
is_trunc_link = leftover_map["name"] != leftover_map["group"]
leftover_map.loc[is_trunc_link, "method"] = leftover_map.loc[is_trunc_link, "method"] + "+4_trunc_link"

post_main_id = (
    post.groupby(["name", "product_id"]).size().reset_index(name="n")
    .sort_values("n", ascending=False).drop_duplicates("name")
    .rename(columns={"product_id": "series_id"})
)
post_main_id["method"] = np.where(post_main_id["name"].map(post_name_n_ids) > 1, "0_id_era_multi_id", "0_id_era")

name_to_series = pd.concat([
    post_main_id[["name", "series_id", "method"]],
    step1_map, step3_map,
    leftover_map[["name", "series_id", "method"]],
], ignore_index=True)

# a name seen in both eras keeps its ID-era row; its pre-ID method is recorded separately
pre_method = pd.concat([step1_map, step3_map, leftover_map])[["name", "method"]].drop_duplicates("name")
name_to_series = name_to_series.drop_duplicates("name", keep="first").merge(
    pre_method.rename(columns={"method": "pre_id_method"}), on="name", how="left"
)
print(f"{len(name_to_series)} names → {name_to_series['series_id'].nunique()} series (before same-day overlap check)")
name_to_series["pre_id_method"].fillna("(ID era only)").value_counts().to_frame("names")

8754 names → 7018 series (before same-day overlap check)


,names
pre_id_method,
1_exact,4749
(ID era only),1922
6_own_series,1271
3b_truncated_prefix,554
6_own_series+4_trunc_link,165
3c_normalized,64
5_handover_short_gap,12
3d_word_order,9
5_handover_long_gap,7


### One name per series per day

Two different pre-ID names that map to the same series **on the same day** were two separate listings
(e.g. `Yumoş Extra Manolya Yumuşatıcı 1440 Ml` at 99.90 and `Yumoş Extra Yumuşatıcı Manolya 1440 Ml` at 199.90
on 2026-02-28). Per series, names are kept in priority order (method rank, then number of rows); a name that
overlaps in time with an already kept name is demoted to its own series.

In [26]:
METHOD_RANK = {"1_exact": 0, "3b_truncated_prefix": 1, "3c_normalized": 2, "3d_word_order": 3,
               "5_handover_short_gap": 4, "5_handover_long_gap": 5, "6_own_series": 6}

pre_map = pd.concat([step1_map, step3_map, leftover_map])[["name", "series_id", "method"]].drop_duplicates("name")
pre_map["rank"] = pre_map["method"].str.split("+").str[0].map(METHOD_RANK)
pre_map["rows"] = pre_map["name"].map(pre.groupby("name").size())
pre_map = pre_map.sort_values(["series_id", "rank", "rows"], ascending=[True, True, False])

demoted = []
for sid, group in pre_map.groupby("series_id"):
    if len(group) == 1:
        continue
    kept_dates = set()
    for name in group["name"]:
        dates = days_by_name[name]
        if kept_dates & dates:
            demoted.append(name)
        else:
            kept_dates |= dates

is_demoted = pre_map["name"].isin(demoted)
demoted_methods = pre_map.loc[is_demoted, "method"].value_counts()
pre_map.loc[is_demoted, "series_id"] = [f"d_{k:04d}" for k in range(int(is_demoted.sum()))]
pre_map.loc[is_demoted, "method"] = "6_own_series (demoted overlap)"

print(f"Names demoted to their own series because of same-day overlap: {len(demoted)}")
demoted_methods.to_frame("demoted names by original method")

Names demoted to their own series because of same-day overlap: 2


,demoted names by original method
method,
3d_word_order,2


In [27]:
panel = raw.merge(pre_map.rename(columns={"series_id": "pre_series_id", "method": "pre_method"}), on="name", how="left")
panel["series_id"] = panel["product_id"].fillna(panel["pre_series_id"])
panel["method"] = np.where(panel["product_id"].notna(), "0_id_era", panel["pre_method"])
assert panel["series_id"].notna().all()

name_to_series = (
    pd.concat([post_main_id[["name", "series_id", "method"]].assign(era="id_era"),
               pre_map[["name", "series_id", "method"]].assign(era="pre_id")], ignore_index=True)
)
name_to_series.to_csv(OUT_DIR / "gurmar_name_to_series.csv", index=False, encoding="utf-8-sig")
panel["method_family"] = panel["method"].str.split("+").str[0]
pre_mask = panel["date"] < ID_START

coverage = pd.DataFrame({
    "pre-ID rows": panel[pre_mask]["method_family"].value_counts(),
    "all rows": panel["method_family"].value_counts(),
}).fillna(0).astype(int)
coverage["pre-ID %"] = (100 * coverage["pre-ID rows"] / pre_mask.sum()).round(1)
coverage

,pre-ID rows,all rows,pre-ID %
method_family,,,
0_id_era,0,217913,0.0
1_exact,272720,272720,80.6
3b_truncated_prefix,555,555,0.2
3c_normalized,3480,3480,1.0
3d_word_order,395,395,0.1
5_handover_long_gap,419,419,0.1
5_handover_short_gap,708,708,0.2
6_own_series,60175,60175,17.8
6_own_series (demoted overlap),72,72,0.0


In [28]:
is_real = ~panel["series_id"].str.startswith(("n_", "d_"))
series = panel.groupby("series_id")["date"].agg(first="min", last="max", days="nunique")
real = series[~series.index.str.startswith(("n_", "d_"))]

pd.Series({
    "real-ID series": len(real),
    "real-ID series reaching back before 2026-05-07": int((real["first"] < ID_START).sum()),
    "median days per real-ID series": real["days"].median(),
    "own series (no ID found or demoted)": int(series.index.str.startswith(("n_", "d_")).sum()),
    "rows on a real ID": f"{is_real.mean():.1%}",
    "pre-ID rows on a real ID": f"{is_real[pre_mask].mean():.1%}",
}, name="value").to_frame()

,value
real-ID series,5757
real-ID series reaching back before 2026-05-07,4801
median days per real-ID series,95.0
own series (no ID found or demoted),1273
rows on a real ID,89.2%
pre-ID rows on a real ID,82.2%


## 7. Quality diagnostics (paper-aligned)

Nothing is removed here; these are the inputs for the next step (cleaning). Paper rules referenced:
duplicates within a store are collapsed by **averaging** (IV-A); product–store changes with
**|Δ| > 80 %** are treated as scraping or unit-parsing artifacts (IV-A). The paper lists alternative outlier
thresholds as future work, so a sensitivity table is reported.

In [29]:
dup = panel.groupby(["series_id", "date"]).agg(n=("price", "size"), n_prices=("price", "nunique"),
                                                method=("method_family", "first"))
dup = dup[dup["n"] > 1]

pd.Series({
    "(series, date) cells with >1 row": len(dup),
    "  … all rows same price": int((dup["n_prices"] == 1).sum()),
    "  … different prices (paper: average)": int((dup["n_prices"] > 1).sum()),
    "rows involved": int(dup["n"].sum()),
}, name="value").to_frame()

,value
"(series, date) cells with >1 row",262
… all rows same price,0
… different prices (paper: average),262
rows involved,529


In [30]:
dup.groupby("method").size().to_frame("duplicate cells by method").sort_values("duplicate cells by method", ascending=False)

,duplicate cells by method
method,
6_own_series,82
5_handover_short_gap,70
1_exact,59
5_handover_long_gap,50
3b_truncated_prefix,1


In [31]:
daily = (
    panel.groupby(["series_id", "date"], as_index=False)
    .agg(price=("price", "mean"), method=("method_family", "first"))
    .sort_values(["series_id", "date"])
)
daily["prev_price"] = daily.groupby("series_id")["price"].shift(1)
daily["change"] = daily["price"] / daily["prev_price"] - 1
changes = daily.dropna(subset=["change"])

THRESHOLDS = [0.2, 0.5, 0.8, 1.0, 2.0]
pd.DataFrame({
    "observations with |Δ| > thr": [int((changes["change"].abs() > t).sum()) for t in THRESHOLDS],
    "share of all changes %": [round(100 * (changes["change"].abs() > t).mean(), 3) for t in THRESHOLDS],
    "share of non-zero changes %": [round(100 * (changes.loc[changes["change"] != 0, "change"].abs() > t).mean(), 2) for t in THRESHOLDS],
}, index=[f"{int(t * 100)}%" for t in THRESHOLDS])

,observations with |Δ| > thr,share of all changes %,share of non-zero changes %
20%,13081,2.382,59.52
50%,1847,0.336,8.40
80%,607,0.111,2.76
100%,227,0.041,1.03
200%,18,0.003,0.08


**Link-boundary check.** For every pre-ID name linked to an ID, compare its last price with the series' next price
after it. Links that cross the outage window span ~3.5 months of inflation, so compare each method with
`1_exact` *within the same `crosses_outage` group*. A good linking method should look like the exact matches (mostly unchanged prices); a bad one shows
many large jumps. This is an automatic proxy for match precision.

In [32]:
linked = panel[pre_mask & is_real].groupby(["name", "method_family", "series_id"]).agg(
    last_date=("date", "max"), last_price=("price", "last")
).reset_index()

series_daily = daily.set_index(["series_id", "date"])["price"]
series_dates = daily.groupby("series_id")["date"].apply(lambda s: s.to_numpy())

def next_price(row):
    dates = series_dates[row["series_id"]]
    after = dates[dates > row["last_date"]]
    return series_daily[(row["series_id"], after[0])] if len(after) else np.nan

linked["next_price"] = linked.apply(next_price, axis=1)
linked["boundary_change"] = linked["next_price"] / linked["last_price"] - 1
linked["crosses_outage"] = linked["last_date"] < OUTAGE_START
lb = linked.dropna(subset=["boundary_change"])

(
    lb.groupby(["crosses_outage", "method_family"])["boundary_change"]
    .agg(links="size",
         unchanged=lambda s: np.isclose(s, 0).mean(),
         abs_gt_20=lambda s: (s.abs() > 0.2).mean(),
         abs_gt_80=lambda s: (s.abs() > 0.8).mean(),
         median_abs=lambda s: s.abs().median())
    .round(3)
)

links  unchanged  abs_gt_20  abs_gt_80  median_abs
crosses_outage method_family                                                           
False          1_exact                 275      0.931      0.025      0.000       0.000
True           1_exact                4474      0.397      0.248      0.015       0.083
               3b_truncated_prefix     554      0.995      0.002      0.000       0.000
               3c_normalized            64      0.453      0.188      0.000       0.049
               3d_word_order             7      0.143      0.571      0.000       0.250
               5_handover_long_gap       7      0.714      0.143      0.000       0.000
               5_handover_short_gap     13      0.462      0.000      0.000       0.056

### COICOP category (repo mapping)

Gurmar's own `kategori` exists only on 2026-02-24 … 2026-05-20. It is mapped with the repo's
`GURMAR_TO_TUIK_MAP` (`Inflations/Codes/Markets/Gurmar/gurmar_tuik_config.py`), using **COICOP 2018** codes
as in the paper's Table IV (personal care = **13**). Series without any `kategori` row stay `unknown`
instead of the repo's silent default to `01`.

In [33]:
GURMAR_TO_COICOP = {
    "Meyve ve Sebze": "01", "Et ve Tavuk": "01", "Süt, Kahvaltılık, Sark.": "01", "Temel Gıda": "01",
    "İçecekler": "01", "Atıştırmalıklar": "01",
    "Bebek Ürünleri": "13",      # repo: "12" (pre-2018 numbering); COICOP 2018 personal care = 13
    "Kişisel Bakım": "13",       # repo: "12"
    "Deterjan ve Temizlik": "05", "Ev ve Yaşam": "05",
    "Kitap, Kırtasiye": "09", "Petshop": "09",
}

cat = panel.dropna(subset=["category"]).groupby("series_id")["category"].agg(lambda s: s.mode().iat[0])
series_cat = pd.DataFrame({"store_category": cat})
series_cat["coicop"] = series_cat["store_category"].map(GURMAR_TO_COICOP)
series_cat = series_cat.reindex(panel["series_id"].unique())
series_cat["coicop"] = series_cat["coicop"].fillna("unknown")

n_cats = panel.dropna(subset=["category"]).groupby("series_id")["category"].nunique()
print(f"Series with >1 store category over time: {int((n_cats > 1).sum())}")
series_cat["coicop"].value_counts().to_frame("series").assign(pct=lambda d: (100 * d["series"] / d["series"].sum()).round(1))

Series with >1 store category over time: 0


,series,pct
coicop,,
01,3688,52.5
13,1349,19.2
unknown,986,14.0
05,910,12.9
09,97,1.4


## 8. Manual validation sample

The paper (Sec. VI) notes that no formal manual validation of the rule-based matching was reported.
A stratified sample per method is exported; fill `is_correct` (1/0) in the CSV and re-run the next cell to get
precision per method with a Wilson 95 % interval.

In [34]:
VALIDATION_FILE = OUT_DIR / "gurmar_match_validation.csv"
PER_METHOD = 30

id_name_list = candidates.groupby("product_id")["name"].apply(lambda s: " | ".join(s))
sample = (
    lb[lb["method_family"] != "0_id_era"]
    .sample(frac=1, random_state=SEED)
    .groupby("method_family").head(PER_METHOD)
)
sample = sample.assign(
    id_names=sample["series_id"].map(id_name_list),
    pack_old=sample["name"].map(parse_pack).astype(str),
    is_correct="",
)[["method_family", "name", "series_id", "id_names", "pack_old", "last_price", "next_price", "boundary_change", "is_correct"]]

if VALIDATION_FILE.exists():
    print(f"{VALIDATION_FILE} exists — not overwritten (it may hold your labels).")
else:
    sample.to_csv(VALIDATION_FILE, index=False, encoding="utf-8-sig")
    print(f"Wrote {len(sample)} rows to {VALIDATION_FILE}")
sample["method_family"].value_counts().to_frame("rows to label")

Wrote 117 rows to data\processed\gurmar_match_validation.csv


,rows to label
method_family,
1_exact,30
3b_truncated_prefix,30
3c_normalized,30
5_handover_short_gap,13
5_handover_long_gap,7
3d_word_order,7


In [35]:
def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return (centre - half, centre + half)

labels = pd.read_csv(VALIDATION_FILE, encoding="utf-8-sig")
labels = labels[pd.to_numeric(labels["is_correct"], errors="coerce").notna()]

if labels.empty:
    print("No labels yet — fill is_correct (1/0) in", VALIDATION_FILE)
else:
    res = labels.groupby("method_family")["is_correct"].agg(n="size", correct="sum")
    res["precision"] = res["correct"] / res["n"]
    res[["ci_low", "ci_high"]] = [wilson(k, n) for k, n in zip(res["correct"], res["n"])]
    display(res.round(3))

No labels yet — fill is_correct (1/0) in data\processed\gurmar_match_validation.csv


## 9. Save the panel

In [36]:
panel_out = panel[["series_id", "date", "price", "name", "product_id", "method", "category", "is_truncated"]].merge(
    series_cat[["coicop"]], left_on="series_id", right_index=True, how="left"
)
panel_out["in_outage"] = panel_out["date"].between(OUTAGE_START, OUTAGE_END)
panel_out.to_parquet(OUT_DIR / "gurmar_panel_mapped.parquet", index=False)
print(f"Saved {len(panel_out):,} rows, {panel_out['series_id'].nunique():,} series → {OUT_DIR / 'gurmar_panel_mapped.parquet'}")

Saved 556,437 rows, 7,030 series → data\processed\gurmar_panel_mapped.parquet


## 10. Regular prices only (drop discounted observations)

The paper has no sale treatment (it compares endpoint snapshots). The project spec keeps **regular prices only**.
Gurmar files carry no sale flag, so discounts are identified from the price path, following the
regular-price literature (Nakamura & Steinsson 2008, *Sales Filter A* / V-shaped sales; Kehoe & Midrigan 2015):

1. **Same name, same day, two prices** → the lower one is the discounted price; keep the higher.
2. **Explicit old price** (`eski_fiyat`, only in the 2026-02-24 file): a row whose old price is above its price is a discount.
3. **V-shaped temporary sale**: the price drops below the current regular price and **returns to at least that level
   within `SALE_WINDOW_DAYS`**. The low days are sale days and are **dropped**. A drop that never returns (or whose
   return falls across the outage window) is a regular price cut.
4. A drop with fewer than `RESOLVE_MIN_DAYS` (21) of data after it cannot be confirmed: marked `sale_unresolved`.
   These rows are kept in the panel but **excluded from test targets** (they fall in the final holdout).

**Leakage note:** step 3 looks into the future (it needs the return). It is fine for defining the target series,
but features at day *t* must not use sale flags that would only be known after *t* — handled in feature engineering.

In [37]:
SALE_WINDOW_DAYS = 35
RESOLVE_MIN_DAYS = 21  # 90% of sale episodes last ≤ 16 days; a drop with ≥ 21 days of data after it and no return is a regular cut
TOL = 0.005  # relative tolerance for "same price"

obs = panel[["series_id", "date", "price", "name", "old_price", "method"]].merge(
    series_cat[["coicop"]], left_on="series_id", right_index=True, how="left"
)
obs["old_price_num"] = obs["old_price"].map(parse_price)
obs["explicit_sale"] = obs["old_price_num"] > obs["price"] * (1 + TOL)

cell = obs.groupby(["series_id", "date"])
daily_obs = (
    obs.assign(rank=cell["price"].rank(method="first", ascending=False))
    .query("rank == 1")
    .drop(columns="rank")
    .sort_values(["series_id", "date"])
    .reset_index(drop=True)
)
n_dup_low = len(obs) - len(daily_obs)

pd.Series({
    "observations (series, date) before": len(obs),
    "same-day lower duplicates dropped (rule 1)": n_dup_low,
    "explicit-sale rows (rule 2)": int(daily_obs["explicit_sale"].sum()),
}, name="value").to_frame()

,value
"observations (series, date) before",556437
same-day lower duplicates dropped (rule 1),267
explicit-sale rows (rule 2),638


In [38]:
def crosses_outage(d0, d1):
    return d0 < OUTAGE_START and d1 > OUTAGE_END

def flag_sales(dates, prices, window=SALE_WINDOW_DAYS, tol=TOL):
    """V-shaped sale filter. Returns (sale, unresolved) boolean arrays."""
    n = len(prices)
    sale = np.zeros(n, dtype=bool)
    unresolved = np.zeros(n, dtype=bool)
    if n == 0:
        return sale, unresolved
    ref = prices[0]
    i = 1
    while i < n:
        p = prices[i]
        if p < ref * (1 - tol):
            j, found, blocked = i + 1, -1, False
            while j < n and (dates[j] - dates[i]).days <= window:
                if crosses_outage(dates[j - 1], dates[j]):
                    blocked = True
                    break
                if prices[j] >= ref * (1 - tol):
                    found = j
                    break
                j += 1
            if found >= 0:
                sale[i:found] = True
                ref = prices[found]
                i = found + 1
                continue
            if not blocked and j >= n and (dates[-1] - dates[i]).days < RESOLVE_MIN_DAYS:
                unresolved[i:] = True
        ref = p
        i += 1
    return sale, unresolved

def apply_filter(frame, window):
    sale_flags, unresolved_flags = [], []
    for _, g in frame.groupby("series_id", sort=False):
        s_, u_ = flag_sales(g["date"].to_list(), g["price"].to_numpy(), window=window)
        sale_flags.append(s_)
        unresolved_flags.append(u_)
    return np.concatenate(sale_flags), np.concatenate(unresolved_flags)

daily_obs["is_sale"], daily_obs["sale_unresolved"] = apply_filter(daily_obs, SALE_WINDOW_DAYS)
daily_obs["is_sale"] |= daily_obs["explicit_sale"]

def change_rate(frame):
    f = frame.sort_values(["series_id", "date"])
    prev = f.groupby("series_id")["price"].shift(1)
    return float(((f["price"] / prev - 1).abs() > TOL)[prev.notna()].mean())

regular = daily_obs[~daily_obs["is_sale"]].copy()

pd.Series({
    "sale observations dropped": int(daily_obs["is_sale"].sum()),
    "share of observations": f"{daily_obs['is_sale'].mean():.2%}",
    "series with ≥1 sale": int(daily_obs.groupby("series_id")["is_sale"].any().sum()),
    "unresolved drops kept (end of data)": int(daily_obs["sale_unresolved"].sum()),
    "price-change rate, shelf price": f"{change_rate(daily_obs):.2%}",
    "price-change rate, regular price": f"{change_rate(regular):.2%}",
}, name="value").to_frame()

,value
sale observations dropped,67268
share of observations,12.09%
series with ≥1 sale,3303
unresolved drops kept (end of data),6512
"price-change rate, shelf price",3.99%
"price-change rate, regular price",2.00%


In [39]:
sens = []
for window in [14, 21, 35, 60]:
    s_, _ = apply_filter(daily_obs, window)
    s_ |= daily_obs["explicit_sale"].to_numpy()
    sens.append({"window_days": window, "sale_obs": int(s_.sum()), "share_%": round(100 * s_.mean(), 2),
                 "change_rate_regular_%": round(100 * change_rate(daily_obs[~s_]), 2)})
pd.DataFrame(sens)

,window_days,sale_obs,share_%,change_rate_regular_%
0,14,20290,3.65,3.07
1,21,62964,11.32,2.10
2,35,67268,12.09,2.00
3,60,68737,12.36,1.96


In [40]:
f = daily_obs.sort_values(["series_id", "date"])
start = f["is_sale"] & ~f.groupby("series_id")["is_sale"].shift(1, fill_value=False)
episode = start.groupby(f["series_id"]).cumsum().where(f["is_sale"])
episodes = f[f["is_sale"]].assign(ep=episode[f["is_sale"]]).groupby(["series_id", "ep"]).agg(
    start=("date", "min"), end=("date", "max"), obs=("date", "size"))
episodes["days"] = (episodes["end"] - episodes["start"]).dt.days + 1

print(f"Sale episodes: {len(episodes)}; median length {episodes['days'].median():.0f} days")
print("Episode length (days):", episodes["days"].describe(percentiles=[.25, .5, .75, .9]).round(1).to_dict())
episodes["start"].dt.day.value_counts().sort_index().to_frame("sale episodes starting on day-of-month").T

Sale episodes: 6710; median length 14 days
Episode length (days): {'count': 6710.0, 'mean': 10.7, 'std': 6.4, 'min': 1.0, '25%': 4.0, '50%': 14.0, '75%': 15.0, '90%': 16.0, 'max': 36.0}


start,1,2,3,4,5,6,7,8,9,10,...,22,23,24,25,26,27,28,29,30,31
sale episodes starting on day-of-month,1845,66,110,141,88,37,24,97,96,62,...,41,210,691,70,63,155,75,18,17,7


In [41]:
example_id = daily_obs.loc[daily_obs["name"].str.contains("8 Kek Mini Orman", na=False), "series_id"].iloc[0]
ex = daily_obs[daily_obs["series_id"] == example_id][["date", "name", "price", "is_sale"]]
ex[ex["date"] < "2026-05-06"].head(40).T

,109895,109896,109897,109898,109899,109900,109901,109902,109903,109904,...,109925,109926,109927,109928,109929,109930,109931,109932,109933,109934
date,2026-02-21 00:00:00,2026-02-22 00:00:00,2026-02-24 00:00:00,2026-02-26 00:00:00,2026-02-27 00:00:00,2026-02-28 00:00:00,2026-03-01 00:00:00,2026-03-02 00:00:00,2026-03-03 00:00:00,2026-03-04 00:00:00,...,2026-03-26 00:00:00,2026-03-27 00:00:00,2026-03-28 00:00:00,2026-03-29 00:00:00,2026-03-30 00:00:00,2026-03-31 00:00:00,2026-04-01 00:00:00,2026-04-02 00:00:00,2026-04-03 00:00:00,2026-04-04 00:00:00
name,8 Kek Mini Orman Meyveli 162 Gr,8 Kek Mini Orman Meyveli 162 Gr,8 Kek Mini Orman Meyveli 162 Gr,8 Kek Mini Orman Meyveli 162 Gr,8 Kek Mini Orman Meyveli 162 Gr,8 Kek Mini Orman Meyveli 162 Gr,8 Kek Mini Orman Meyveli 162 Gr,8 Kek Mini Orman Meyveli 162 Gr,8 Kek Mini Orman Meyveli 162 Gr,8 Kek Mini Orman Meyveli 162 Gr,...,8 Kek Mini Orman Meyveli 162 Gr,8 Kek Mini Orman Meyveli 162 Gr,8 Kek Mini Orman Meyveli 162 Gr,8 Kek Mini Orman Meyveli 162 Gr,8 Kek Mini Orman Meyveli 162 Gr,8 Kek Mini Orman Meyveli 162 Gr,8 Kek Mini Orman Meyveli 162 Gr,8 Kek Mini Orman Meyveli 162 Gr,8 Kek Mini Orman Meyveli 162 Gr,8 Kek Mini Orman Meyveli 162 Gr
price,55.0,55.0,55.0,55.0,55.0,55.0,47.5,47.5,47.5,47.5,...,55.0,55.0,55.0,55.0,55.0,55.0,41.95,41.95,41.95,41.95
is_sale,False,False,False,False,False,False,True,True,True,True,...,False,False,False,False,False,False,True,True,True,True


## 11. Unit price (paper Eq. 4)

Paper Sec. IV-C, Eq. 4: $p^{std}_i = p_i \cdot 1/q_i$ — package price divided by package quantity, used there for the
hunger-threshold basket. Here: price per **kg** (solids), per **litre** (liquids), or per **piece** (count-only packs).

**Does predicting the unit price help?** For one product with a fixed pack, unit price = price / constant, so its
percentage (or log) change is **identical** to the price change — no gain when the target is a return. It matters when:
- the **pack size changes under the same series** (shrinkflation: 800 g → 750 g at the same price is a +6.7 % unit-price
  increase that a shelf-price series shows as 0 %);
- building **cross-product features** (a product's unit price relative to its category median is comparable across sizes);
- a model predicts **price levels** (scale problem) — avoided by predicting returns.

In [42]:
pack_cache = {n: parse_pack(n) for n in regular["name"].unique()}
regular[["pack_count", "pack_size", "pack_unit"]] = pd.DataFrame(
    [pack_cache[n] for n in regular["name"]], index=regular.index
)

qty = regular["pack_count"].fillna(1) * regular["pack_size"]
regular["unit_price"] = np.where(
    regular["pack_size"].notna(), regular["price"] / qty * 1000,
    np.where(regular["pack_count"].notna(), regular["price"] / regular["pack_count"], np.nan),
)
regular["unit_basis"] = np.select(
    [regular["pack_unit"].eq("g"), regular["pack_unit"].eq("ml"), regular["pack_count"].notna()],
    ["TRY/kg", "TRY/l", "TRY/piece"], default="none",
)

basis = regular.groupby("unit_basis").agg(rows=("price", "size"), series=("series_id", "nunique"))
basis["rows_%"] = (100 * basis["rows"] / basis["rows"].sum()).round(1)
basis

,rows,series,rows_%
unit_basis,,,
TRY/kg,252059,3582,51.6
TRY/l,140020,2045,28.6
TRY/piece,49868,735,10.2
none,46955,1298,9.6


In [43]:
def pack_key(count, size, unit):
    if pd.isna(size) and pd.isna(count):
        return np.nan
    count = 1 if pd.isna(count) else int(count)
    return f"{count}x{size:g}{unit}" if not pd.isna(size) else f"{count}pc"

regular["pack_key"] = [pack_key(c, s_, u) for c, s_, u in
                       zip(regular["pack_count"], regular["pack_size"], regular["pack_unit"])]

keys_per_series = regular.dropna(subset=["pack_key"]).groupby("series_id")["pack_key"].nunique()
pack_changed_series = keys_per_series[keys_per_series > 1].index
regular = regular.sort_values(["series_id", "date"])
_key = regular.groupby("series_id")["pack_key"].transform(lambda s: s.ffill().bfill()).fillna("")
_new = _key.ne(_key.groupby(regular["series_id"]).shift()) & regular.groupby("series_id").cumcount().gt(0)
regular["pack_segment"] = _new.astype(int).groupby(regular["series_id"]).cumsum() + 1  # same rule as src/preprocessing/features.py
print(f"Pack segments: {regular[['series_id', 'pack_segment']].drop_duplicates().shape[0]:,} for "
      f"{regular['series_id'].nunique():,} series")

r = regular.sort_values(["series_id", "date"])
log_ret_price = np.log(r["price"]).groupby(r["series_id"]).diff()
log_ret_unit = np.log(r["unit_price"]).groupby(r["series_id"]).diff()
same_pack_rows = ~r["series_id"].isin(pack_changed_series) & log_ret_unit.notna()

pd.Series({
    "series with a pack-size change": len(pack_changed_series),
    "max |Δlog price − Δlog unit price| where the pack never changes": float((log_ret_price - log_ret_unit)[same_pack_rows].abs().max()),
    "pack-change days where shelf price is flat but unit price moves": int(
        ((log_ret_price.abs() < 1e-9) & (log_ret_unit.abs() > 1e-9) & r["series_id"].isin(pack_changed_series)).sum()),
}, name="value").to_frame()

Pack segments: 7,291 for 7,025 series


,value
series with a pack-size change,1.690000e+02
max |Δlog price − Δlog unit price| where the pack never changes,1.776357e-15
pack-change days where shelf price is flat but unit price moves,2.250000e+02


In [44]:
shrink = r[r["series_id"].isin(pack_changed_series)].copy()
shrink["prev_key"] = shrink.groupby("series_id")["pack_key"].shift(1)
switches = shrink[shrink["pack_key"].notna() & shrink["prev_key"].notna() & (shrink["pack_key"] != shrink["prev_key"])]
switches[["series_id", "date", "name", "price", "prev_key", "pack_key", "unit_price"]].head(12)

,series_id,date,name,price,prev_key,pack_key,unit_price
1599,10027,2026-08-28,Cin Lokmalık 114 Gr 18'li,25.00,1x114g,18x114g,12.183236
1619,10027,2026-09-17,Eti Cin Lokmalık 114 Gr,25.00,18x114g,1x114g,219.298246
5921,10130,2026-10-02,Ülker Tuzlu Çubuk Kraker 40 Gr,7.50,1x32g,1x40g,187.500000
7208,10147,2026-08-28,Klasik Yeşil Düz Bulaşık Süngeri 10 Adet,99.90,5pc,10pc,9.990000
7215,10147,2026-09-04,Parex Klasik Yeşil Düz Bulaşık Süngeri 5Li,99.90,10pc,5pc,19.980000
14843,10509,2026-08-28,Danone Kakaolu Süt 27'li Paket,29.95,1x180ml,27pc,1.109259
14862,10509,2026-09-16,Danone Kakaolu Süt 180 Ml,29.95,27pc,1x180ml,166.388889
19677,10607,2026-08-28,Bebe Sütlü Bisküvi 172 Gr,89.00,1x400g,1x172g,517.441860
19699,10607,2026-09-23,Ülker Bebe Sütlü Bisküvi 400 Gr,89.00,1x172g,1x400g,222.500000
20881,10631,2026-10-02,Ülker 9 Kat Ince Ince Fındıklı 114 Gr,29.95,1x144g,1x114g,262.719298


## 12. Train / test blocks

Data segments: **A** full catalogue 2026-02-21 → 2026-05-04 (70 scrape days) · **outage** 2026-05-06 → 2026-08-17
(275 products) · **B** full catalogue 2026-08-18 → 2026-10-04 (48 scrape days).

Three expanding-window blocks; every test window contains target days (1st, 15th, last day of month, holidays):

| Block | Train | Test | Purpose |
|---|---|---|---|
| B1 | 02-21 → 04-04 | 04-05 → 05-04 | development, segment A only |
| B2 | 02-21 → 08-31 (outage excluded; 08-18 → 08-31 = burn-in to rebuild lags after the gap) | 09-01 → 09-17 | development, across the gap |
| B3 | 02-21 → 09-17 (outage excluded) | 09-18 → 10-04 | **final holdout — evaluate once** |

Outage days are excluded from train targets by default; the 275 products with continuous outage data can be
added later as an experiment (`USE_OUTAGE_ROWS`).

In [45]:
import json

HOLIDAYS_2026 = pd.to_datetime([
    "2026-01-01", "2026-03-19", "2026-03-20", "2026-03-21", "2026-03-22",   # Ramazan Bayramı (arife + 3 days)
    "2026-04-23", "2026-05-01", "2026-05-19",
    "2026-05-26", "2026-05-27", "2026-05-28", "2026-05-29", "2026-05-30",   # Kurban Bayramı (arife + 4 days)
    "2026-07-15", "2026-08-30", "2026-10-28", "2026-10-29",
])

def is_target_day(dates):
    dates = pd.DatetimeIndex(dates)
    return (dates.day == 1) | (dates.day == 15) | dates.is_month_end | dates.isin(HOLIDAYS_2026)

USE_OUTAGE_ROWS = False
BLOCKS = [
    {"block": "B1", "train_start": "2026-02-21", "train_end": "2026-04-04", "test_start": "2026-04-05", "test_end": "2026-05-04"},
    {"block": "B2", "train_start": "2026-02-21", "train_end": "2026-08-31", "test_start": "2026-09-01", "test_end": "2026-09-17",
     "burn_in": ["2026-08-18", "2026-08-31"]},
    {"block": "B3", "train_start": "2026-02-21", "train_end": "2026-09-17", "test_start": "2026-09-18", "test_end": "2026-10-04",
     "final_holdout": True},
]

regular = regular.sort_values(["series_id", "date"])
regular["prev_date"] = regular.groupby("series_id")["date"].shift(1)
regular["prev_price"] = regular.groupby("series_id")["price"].shift(1)
regular["in_outage"] = regular["date"].between(OUTAGE_START, OUTAGE_END)
regular["is_target_day"] = is_target_day(regular["date"])

rows = []
for b in BLOCKS:
    train = regular[regular["date"].between(b["train_start"], b["train_end"])]
    if not USE_OUTAGE_ROWS:
        train = train[~train["in_outage"]]
    test_all = regular[regular["date"].between(b["test_start"], b["test_end"])]
    test = test_all[~test_all["sale_unresolved"]]
    one_step = test[(test["date"] - test["prev_date"]).dt.days.le(3)]
    changed = (one_step["price"] / one_step["prev_price"] - 1).abs() > TOL
    test_days = pd.DatetimeIndex(sorted(test["date"].unique()))
    rows.append({
        "block": b["block"],
        "train": f"{b['train_start'][5:]} → {b['train_end'][5:]}",
        "test": f"{b['test_start'][5:]} → {b['test_end'][5:]}",
        "train_days": train["date"].nunique(), "train_rows": len(train), "train_series": train["series_id"].nunique(),
        "test_days": len(test_days), "test_rows": len(test), "test_series": test["series_id"].nunique(),
        "test_rows_excluded_unresolved": int(test_all["sale_unresolved"].sum()),
        "one_step_test_rows": len(one_step), "test_change_rate_%": round(100 * changed.mean(), 2),
        "target_days_in_test": ", ".join(d.strftime("%m-%d") for d in test_days[is_target_day(test_days)]),
    })

blocks_summary = pd.DataFrame(rows).set_index("block")
(OUT_DIR / "gurmar_blocks.json").write_text(json.dumps(
    {"blocks": BLOCKS, "outage": [str(OUTAGE_START.date()), str(OUTAGE_END.date())],
     "use_outage_rows": USE_OUTAGE_ROWS, "sale_window_days": SALE_WINDOW_DAYS}, indent=2))
blocks_summary.T

block,B1,B2,B3
train,02-21 → 04-04,02-21 → 08-31,02-21 → 09-17
test,04-05 → 05-04,09-01 → 09-17,09-18 → 10-04
train_days,40,83,100
train_rows,173002,349553,404694
train_series,5863,6693,6908
test_days,29,17,17
test_rows,122928,54534,55137
test_series,5602,4725,4321
test_rows_excluded_unresolved,500,607,4859
one_step_test_rows,119978,51775,53436


In [46]:
regular_out = regular.drop(columns=["old_price", "old_price_num", "explicit_sale", "prev_date", "prev_price"])
regular_out.to_parquet(OUT_DIR / "gurmar_panel_regular.parquet", index=False)
print(f"Saved {len(regular_out):,} regular-price rows, {regular_out['series_id'].nunique():,} series "
      f"→ {OUT_DIR / 'gurmar_panel_regular.parquet'}; blocks → {OUT_DIR / 'gurmar_blocks.json'}")

Saved 488,902 regular-price rows, 7,025 series → data\processed\gurmar_panel_regular.parquet; blocks → data\processed\gurmar_blocks.json


## Next

- Feature engineering on `gurmar_panel_regular.parquet` with as-of logic (no sale flags from the future).
- Target: log return of the regular shelf price; split series at pack-size changes (`pack_segment`) or use unit price there.
- Label `gurmar_match_validation.csv` to get matching precision.